# Pháp Điển AI — Đo trên Kaggle (không Zalo/VLSP)

Import notebook này, bật Internet, GPU T4 ×2, Add Input đúng một bộ kết quả OFFLINE cuối.
Code phải có trên origin/main trước khi Run All. Gold 120 là DRAFT, chỉ đo phát triển.
Không cần frontend, ngrok, Neo4j hoặc VN_LABOR_API_KEY.

Chọn PHASE = retrieval / ablation / naive_qa. Bắt đầu PROFILE = smoke.
Sau khi smoke đạt, chuyển PROFILE = standard để đo đủ câu.
QA cần GEMINI_API_KEY và Ollama; HF_TOKEN tùy chọn. Dừng backend/tác vụ khác dùng cùng GPU hoặc quota.
Notebook chỉ chạy các nhánh đã triển khai; không tự bật baseline upstream chưa kiểm chứng.


In [ ]:
from pathlib import Path
REPO_URL = "https://github.com/tvqui/legal-agentic-rag.git"
ROOT = Path('/kaggle/working/legal-agentic-rag')
WORK = Path('/kaggle/working/unified_evaluation')
PHASE = 'retrieval'  # retrieval | ablation | naive_qa
PROFILE = 'smoke'    # smoke = 5 câu; standard = toàn bộ tập đủ điều kiện
ARTIFACT = 'auto'    # hoặc đường dẫn ZIP / thư mục artifacts cụ thể trong /kaggle/input
LABOR_GOLD = 'gold_dataset/PhapDienAI_Gold_120_Labor_v0_2.zip'
GEMINI_MODEL = 'gemini-3.1-flash-lite'
RESEARCHER_PROVIDER = 'gemini'
RESUME = False      # chỉ bật khi giữ đúng code/data/model/settings của run cũ
RETRY_ERRORS = False
# Nếu cần đổi model, dữ liệu hoặc code: đặt tên RUN_NAME mới và RESUME=False.
RUN_NAME = 'labor120_v1'
STAGES = ['retrieval'] if PHASE == 'retrieval' else ['ablation']
if PHASE not in {'retrieval','ablation','naive_qa'}:raise ValueError('PHASE không hợp lệ')
if PROFILE not in {'smoke','standard'}:raise ValueError('PROFILE không hợp lệ')


## Cài môi trường và lấy code đã push


In [ ]:
import os,sys,subprocess,shutil,json,time,urllib.request
WORK.mkdir(parents=True,exist_ok=True)
def run(args, **kw):
    print('RUN',args[0],flush=True)
    return subprocess.run(args,check=True,**kw)
run(['apt-get','update','-y'])
run(['apt-get','install','-y','git','curl','zstd'])
if not ROOT.exists():
    run(['git','clone','--branch','main','--single-branch',REPO_URL,str(ROOT)])
else:
    dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=ROOT,text=True)
    if dirty.strip():raise RuntimeError('Tracked code was edited in the notebook. Save changes before updating.')
    run(['git','fetch','--prune','origin'],cwd=ROOT)
    run(['git','checkout','--detach','origin/main'],cwd=ROOT)
run(['git','log','-1','--oneline'],cwd=ROOT)
# Keep Kaggle's CUDA-compatible Torch and isolate project dependencies.
import importlib.metadata
constraints=[]
for package in ('torch','torchvision','torchaudio'):
    try:constraints.append(package+'=='+importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        if package=='torch':raise RuntimeError('Kaggle runtime has no Torch; select a GPU image.')
constraint_file=WORK/'torch_constraints.txt'
constraint_file.write_text('\n'.join(constraints)+'\n',encoding='utf-8')
VENV=WORK/'.venv'
run([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(VENV)])
PYTHON=str(VENV/'bin/python')
run([sys.executable,'-m','pip','--python',PYTHON,'install','--disable-pip-version-check',
    '-q','-c',str(constraint_file),'-e',str(ROOT)+'[retrieval,online]','datasets'])
run([PYTHON,'-c','import torch,FlagEmbedding,faiss,datasets; assert torch.cuda.is_available(); print("CUDA dependencies ready:",torch.cuda.device_count())'])
if not (ROOT/'benchmarks/evaluation_suite/protocol.py').exists() or 'labor-evaluation-v2-20261028' not in (ROOT/'benchmarks/evaluation_suite/protocol.py').read_text(encoding='utf-8'):
    raise RuntimeError('Push evaluation suite v2 before running this notebook.')
if not (ROOT/'benchmarks/evaluation_suite/suite.py').exists():
    raise RuntimeError('Push the evaluation_suite folder to GitHub before Run All.')

if not (ROOT/'đo/evaluate_project.py').exists():
    raise RuntimeError('Chưa có bộ đo mới trên GitHub. Push folder đo rồi chạy lại cell lấy code.')
run([sys.executable,'-m','pip','--python',PYTHON,'install','--disable-pip-version-check','-q',
     '-c',str(constraint_file),'-r',str(ROOT/'đo/requirements.txt')])


In [ ]:
from kaggle_secrets import UserSecretsClient
client = UserSecretsClient()
try:
    hf_token = client.get_secret('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN: đã nạp' if hf_token else 'HF_TOKEN: không có, dùng tải công khai')
except Exception:
    print('HF_TOKEN: tùy chọn, dùng tải công khai')
if PHASE != 'retrieval':
    key = client.get_secret('GEMINI_API_KEY')
    if not key:raise RuntimeError('Bật Secret GEMINI_API_KEY cho notebook QA.')
    os.environ['GEMINI_API_KEY'] = key
    os.environ['GEMINI_MODEL'] = GEMINI_MODEL
    os.environ['VN_LABOR_GEMINI_RPM'] = '13'
    os.environ['VN_LABOR_RESEARCHER_MODE'] = RESEARCHER_PROVIDER
    os.environ['VN_LABOR_ADJUDICATION_REVIEW_PROVIDER'] = 'gemini'
    os.environ['VN_LABOR_ADJUDICATION_REVIEW_MODEL'] = GEMINI_MODEL
    run([PYTHON,'-u','kaggle/check_gemini.py','--config','config/online_kaggle.yaml',
         '--model',GEMINI_MODEL],cwd=ROOT,timeout=300)
else:
    print('Truy xuất thuần: không cần Gemini/Ollama, không gọi model tạo câu trả lời.')


In [ ]:
if {'ablation','vlsp'} & set(STAGES):
    # Use GPU 1 for Ollama, GPU 0 for embedding/reranker. Own only the server
    # started by this notebook; never terminate another session's server.
    gpu_lines=subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True).splitlines()
    print('GPUs:',gpu_lines,flush=True)
    if len(gpu_lines)<2:raise RuntimeError('Select GPU T4 x2 in Kaggle settings for this configuration.')
    if shutil.which('ollama') is None:
        run(['bash','-lc','set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh'])
    def ollama_json(path):
        with urllib.request.urlopen('http://127.0.0.1:11434'+path,timeout=5) as response:return json.load(response)
    OWN_OLLAMA=None
    OLLAMA_LOG=(WORK/'ollama.log').open('a',encoding='utf-8')
    try:
        ollama_json('/api/tags')
        print('Reusing existing Ollama; verify it is not serving another workload.',flush=True)
    except Exception:
        environment=dict(os.environ);environment['CUDA_VISIBLE_DEVICES']='1'
        OWN_OLLAMA=subprocess.Popen(['ollama','serve'],env=environment,stdout=OLLAMA_LOG,stderr=subprocess.STDOUT)
        for attempt in range(60):
            if OWN_OLLAMA.poll() is not None:raise RuntimeError('Ollama stopped; read '+str(WORK/'ollama.log'))
            try:ollama_json('/api/tags');break
            except Exception:time.sleep(2)
        else:raise RuntimeError('Ollama did not become ready')
    models={m.get('name') for m in ollama_json('/api/tags').get('models',[])}
    if 'qwen3:8b' not in models:
        for attempt in range(3):
            try:run(['ollama','pull','qwen3:8b'],timeout=900);break
            except (subprocess.CalledProcessError,subprocess.TimeoutExpired):
                if attempt==2:raise
                time.sleep(10)
    print('Ollama and model ready.',flush=True)

else:
    OWN_OLLAMA=None;OLLAMA_LOG=None


## Chuẩn bị dữ liệu và chạy phần đã chọn


In [ ]:
import json, subprocess, sys
from datetime import datetime, timezone
OUT = WORK / RUN_NAME / PROFILE
OUT.mkdir(parents=True, exist_ok=True)
BUNDLE = OUT / 'bundle120'
SETTINGS = OUT / 'settings.json'
# Dùng bộ phát hiện Input của suite đã có; không nhận checkpoint/recovery làm corpus.
sys.path.insert(0, str(ROOT/'benchmarks/evaluation_suite'))
from common import detect_artifact
artifact = Path(detect_artifact(Path('/kaggle/input'))) if ARTIFACT == 'auto' else Path(ARTIFACT)
if not artifact.exists():raise FileNotFoundError(artifact)
config = json.loads((ROOT/'đo/settings.example.json').read_text(encoding='utf-8'))
config.update(online_config=str(ROOT/'config/online_kaggle.yaml'), cache=str(WORK/'cache'), device='cuda:0')
config['judge']['model'] = GEMINI_MODEL
if SETTINGS.exists() and json.loads(SETTINGS.read_text(encoding='utf-8')) != config:
    raise RuntimeError('Settings đã thay đổi: đặt RUN_NAME mới, không trộn model/settings vào run cũ.')
SETTINGS.write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
def evaluate(*arguments):
    return subprocess.run([str(PYTHON),'-u',*map(str,arguments)],cwd=ROOT,check=True)
if not (BUNDLE/'manifest.json').exists():
    evaluate('đo/evaluate_project.py','prepare','--artifact',artifact,
             '--gold',ROOT/LABOR_GOLD,'--bundle',BUNDLE,'--cache',WORK/'cache','--allow-draft')
else:
    frozen = json.loads((BUNDLE/'manifest.json').read_text(encoding='utf-8'))
    if Path(frozen['artifact_source']).resolve() != artifact.resolve():
        raise RuntimeError('Input đổi: tạo RUN_NAME mới để chuẩn bị bundle mới.')
    print('Giữ bundle đã đóng băng; runner sẽ kiểm hash code/data/model khi đo hoặc resume.')
print('Input:',artifact)
print('Kết quả:',OUT)


In [ ]:
METHODS = {
    'retrieval': ['bm25','dense','hybrid','hybrid_rerank','hybrid_graph','hybrid_graph_rerank'],
    'ablation': ['full','without_graph','without_reranker','without_auditor','without_review'],
    'naive_qa': ['naive_bm25','naive_dense','naive_hybrid'],
}[PHASE]
SCRIPT = 'đo/evaluate_baselines.py' if PHASE == 'naive_qa' else 'đo/evaluate_project.py'
RUN = OUT / PHASE
repeats = 3 if PHASE == 'retrieval' and PROFILE == 'standard' else 1
args = [SCRIPT,'run','--bundle',BUNDLE,'--config',SETTINGS,'--out',RUN,
        '--task','retrieval' if PHASE == 'retrieval' else 'qa',
        '--methods',*METHODS,'--repeats',str(repeats),'--k','10','--allow-draft']
if PROFILE == 'smoke':args += ['--limit','5']
if PHASE != 'retrieval':args += ['--allow-llm']
if RESUME:args += ['--resume']
if RETRY_ERRORS:args += ['--retry-errors']
evaluate(*args)
# Chỉ chấm sau generation để judge không tiêu quota trước các cấu hình còn lại.
if PHASE != 'retrieval':
    evaluate(SCRIPT,'judge','--run',RUN,'--allow-llm')
# Tổng hợp những run tương thích đã có; khác task sẽ được giữ thành nhóm riêng.
runs = [p for p in OUT.iterdir() if p.is_dir() and (p/'run_manifest.json').exists()]
evaluate('đo/evaluate_project.py','report','--runs',*runs,'--out',OUT/'comparison')


## Tải kết quả — chạy được cả khi cell đo dừng vì quota


In [ ]:
# Cell này có thể chạy riêng sau khi bước đo dừng/lỗi để tải checkpoint.
import shutil
from IPython.display import FileLink, display, Markdown
if 'OUT' not in globals() or not OUT.exists():
    raise RuntimeError('Chạy cell chuẩn bị để xác định OUT trước.')
for table in [OUT/'comparison/comparison.md', OUT/PHASE/'report/comparison.md']:
    if table.exists():
        display(Markdown(table.read_text(encoding='utf-8')))
        break
archive = Path(shutil.make_archive(str(OUT)+'_results','zip',root_dir=OUT))
display(FileLink(str(archive)))
print('Tải ZIP này; có bundle, cấu hình, responses, trace và trạng thái tiến độ. Không chứa venv/model cache.')


## Dọn dịch vụ sau khi kết thúc tất cả các phần


In [ ]:
# Chỉ chạy khi kết thúc toàn bộ phép đo trong phiên này; không chạy giữa các phase.
if OWN_OLLAMA is not None and OWN_OLLAMA.poll() is None:
    OWN_OLLAMA.terminate()
    try:OWN_OLLAMA.wait(timeout=10)
    except subprocess.TimeoutExpired:
        OWN_OLLAMA.kill(); OWN_OLLAMA.wait()
if OLLAMA_LOG is not None: OLLAMA_LOG.close()
print('Đã dọn Ollama do notebook này tạo; không dừng dịch vụ được tái sử dụng.')
